# Book Commercial Success -- Features & Enriching Dataset & EDA

**Datasets:**
- merged_books.csv 
- Open Library API
- GoogleBooks Library API

**Enrich dataset**
1. Add publisher information
2. Add auxilary columns/information

   a. historical_nyt_rate = #NYBT books in Oct / #books in Oct.
   
   b. (Not sure if this is useful)historical_publisher_nyt_rate = #NYBT books in X publisher / #books in X publisher








## 0. Install & Import 

In [3]:
import pandas as pd
import requests
import time
import re
from pathlib import Path
from tqdm import tqdm

## 1. Load Datasets
**Thao_analysis**:
- merged_books.csv -- embedding of nyt_hardcover_fiction_bestsellers into our baseline Goodbooks


In [22]:

df = pd.read_csv("data/merged_books.csv", dtype={"isbn13": str})

df.head()

print(df.shape)
display(df[["title", "authors", "isbn13", "genres"]].head(5))



(9417, 48)


,title,authors,isbn13,genres
0,"The Hunger Games (The Hunger Games, #1)",['Suzanne Collins'],9780439023480.0,"['young-adult', 'fiction', 'fantasy', 'science..."
1,Harry Potter and the Sorcerer's Stone (Harry P...,"['J.K. Rowling', 'Mary GrandPré']",9780439554930.0,"['fantasy', 'fiction', 'young-adult', 'classics']"
2,"Twilight (Twilight, #1)",['Stephenie Meyer'],9780316015840.0,"['young-adult', 'fantasy', 'romance', 'fiction..."
3,To Kill a Mockingbird,['Harper Lee'],9780061120080.0,"['classics', 'fiction', 'historical-fiction', ..."
4,The Fault in Our Stars,['John Green'],9780525478810.0,"['young-adult', 'romance', 'fiction', 'contemp..."


## 2. Update dataset
Remove all the non-fiction books

In [ ]:
def parse_genres(x):
    if pd.isna(x):
        return []
    
    # If already a list, return it
    if isinstance(x, list):
        return x
    
    x = str(x).strip()
    
    # Remove extra outer quotes if they exist
    if (x.startswith('"') and x.endswith('"')) or (x.startswith("'") and x.endswith("'")):
        x = x[1:-1]
    
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return parsed
        else:
            return [parsed]
    except Exception:
        return [x]

def is_nonfiction_book(genres):
    genres = parse_genres(genres)
    
    genres_clean = [
        str(g).lower().strip()
        for g in genres
    ]
    
    return "nonfiction" in genres_clean

In [28]:
df["is_nonfiction"] = df["genres"].apply(is_nonfiction_book)

num_nonfiction = df["is_nonfiction"].sum()
total_books = len(df)
nonfiction_rate = num_nonfiction / total_books

print(f"Number of non-fiction books: {num_nonfiction}")
print(f"Total books: {total_books}")
print(f"Non-fiction rate: {nonfiction_rate:.2%}")

Number of non-fiction books: 0
Total books: 9417
Non-fiction rate: 0.00%


---
## 3. Enrich dataset

**Goals:**
- Use Open Library,GoogleBooks API to add 'publisher_or_imprint' column

In [ ]:
# -- 2a. Setup & Clean ISBN13 ---------------
ISBN_COL = "isbn13"
CACHE_PATH = Path("publisher_cache.csv")

USER_AGENT = "BookSuccessBootcampProject/1.0 (ron.yang228@gmail.com)"

# Optional. Leave as None unless you have a Google Books API key.
GOOGLE_BOOKS_API_KEY = None

def clean_isbn13(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip()
    x = re.sub(r"\.0$", "", x)
    x = re.sub(r"[^0-9]", "", x)
    
    if len(x) == 13:
        return x
    
    return None 

df["isbn13_lookup"] = df[ISBN_COL].apply(clean_isbn13)

df[["title", "isbn13", "isbn13_lookup"]].head(20)



,title,isbn13,isbn13_lookup
0,"The Hunger Games (The Hunger Games, #1)",9780439023480.0,9780439023480
1,Harry Potter and the Sorcerer's Stone (Harry P...,9780439554930.0,9780439554930
2,"Twilight (Twilight, #1)",9780316015840.0,9780316015840
3,To Kill a Mockingbird,9780061120080.0,9780061120080
4,The Fault in Our Stars,9780525478810.0,9780525478810
5,The Hobbit,9780618260300.0,9780618260300
6,The Catcher in the Rye,9780316769170.0,9780316769170
7,"Angels & Demons (Robert Langdon, #1)",9781416524790.0,9781416524790
8,The Kite Runner,9781594480000.0,9781594480000
9,"Divergent (Divergent, #1)",9780062024040.0,9780062024040


In [ ]:
# -- 2b. Open Library Lookup ---------------

def get_openlibrary_publisher(isbn13):
    if isbn13 is None:
        return None
    
    url = f"https://openlibrary.org/isbn/{isbn13}.json"
    headers = {"User-Agent": USER_AGENT}
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        
        if r.status_code != 200:
            return None
        
        data = r.json()
        publishers = data.get("publishers", [])
        
        if isinstance(publishers, list) and len(publishers) > 0:
            return publishers[0]
        
        return None
    
    except Exception:
        return None
    

In [ ]:
# -- 2c. GoogleBook Lookup ---------------
def get_google_books_publisher(isbn13):
    if isbn13 is None:
        return None
    
    url = "https://www.googleapis.com/books/v1/volumes"
    params = {
        "q": f"isbn:{isbn13}",
        "fields": "items(volumeInfo/publisher)"
    }
    
    if GOOGLE_BOOKS_API_KEY is not None:
        params["key"] = GOOGLE_BOOKS_API_KEY
    
    try:
        r = requests.get(url, params=params, timeout=10)
        
        if r.status_code != 200:
            return None
        
        data = r.json()
        items = data.get("items", [])
        
        if len(items) == 0:
            return None
        
        return items[0].get("volumeInfo", {}).get("publisher", None)
    
    except Exception:
        return None

In [ ]:
# -- 2d. Combined Lookup ---------------
def get_publisher_or_imprint(isbn13, sleep_time=0.25):
    publisher = get_openlibrary_publisher(isbn13)
    
    if publisher is not None:
        time.sleep(sleep_time)
        return publisher, "openlibrary"
    
    time.sleep(sleep_time)
    
    publisher = get_google_books_publisher(isbn13)
    
    if publisher is not None:
        time.sleep(sleep_time)
        return publisher, "google_books"
    
    time.sleep(sleep_time)
    return None, "missing"

In [ ]:
# -- 2e. Build Cache ---------------
def build_publisher_cache(df, isbn_col="isbn13_lookup", cache_path=CACHE_PATH):
    unique_isbns = sorted(df[isbn_col].dropna().unique())
    
    if cache_path.exists():
        cache = pd.read_csv(cache_path, dtype={"isbn13_lookup": str})
    else:
        cache = pd.DataFrame(
            columns=["isbn13_lookup", "publisher_or_imprint", "publisher_source"]
        )
    
    already_done = set(cache["isbn13_lookup"].dropna().astype(str))
    missing_isbns = [isbn for isbn in unique_isbns if isbn not in already_done]
    
    print(f"Total unique ISBNs: {len(unique_isbns)}")
    print(f"Already cached: {len(already_done)}")
    print(f"Need to query: {len(missing_isbns)}")
    
    new_rows = []
    
    for isbn in tqdm(missing_isbns):
        publisher, source = get_publisher_or_imprint(isbn)
        
        new_rows.append({
            "isbn13_lookup": isbn,
            "publisher_or_imprint": publisher,
            "publisher_source": source
        })
        
        if len(new_rows) % 100 == 0:
            cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
            cache.to_csv(cache_path, index=False)
            new_rows = []
    
    if len(new_rows) > 0:
        cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
        cache.to_csv(cache_path, index=False)
    
    return cache

In [12]:
# -- 2f. Add publisher column to the dataset ---------------
publisher_cache = build_publisher_cache(df)

df = df.merge(
    publisher_cache,
    on="isbn13_lookup",
    how="left"
)

df["publisher_or_imprint"] = df["publisher_or_imprint"].fillna("Unknown")
df["publisher_source"] = df["publisher_source"].fillna("missing")

Total unique ISBNs: 8589
Already cached: 0
Need to query: 8589


100%|██████████| 8589/8589 [2:23:30<00:00,  1.00s/it]  


In [13]:
print(df.shape)
display(df[["title", "isbn13_lookup", "publisher_or_imprint", "publisher_source"]].head(20))

(9417, 51)


,title,isbn13_lookup,publisher_or_imprint,publisher_source
0,"The Hunger Games (The Hunger Games, #1)",9780439023480,Unknown,missing
1,Harry Potter and the Sorcerer's Stone (Harry P...,9780439554930,Arthur A. Levine Books,openlibrary
2,"Twilight (Twilight, #1)",9780316015840,Unknown,missing
3,To Kill a Mockingbird,9780061120080,Unknown,missing
4,The Fault in Our Stars,9780525478810,Unknown,missing
5,The Hobbit,9780618260300,Houghton Mifflin,openlibrary
6,The Catcher in the Rye,9780316769170,Unknown,missing
7,"Angels & Demons (Robert Langdon, #1)",9781416524790,Unknown,missing
8,The Kite Runner,9781594480000,Unknown,missing
9,"Divergent (Divergent, #1)",9780062024040,Unknown,missing


In [14]:
coverage = (df["publisher_or_imprint"] != "Unknown").mean()
print(f"Publisher coverage: {coverage:.2%}")

df["publisher_source"].value_counts(dropna=False)

Publisher coverage: 9.20%


publisher_source
missing        8551
openlibrary     866
Name: count, dtype: int64

Will not use publisher as information since the bad coverage rate.

# 3. Remove non-fiction books